In [26]:
from elasticsearch import Elasticsearch
# es = Elasticsearch("http://localhost:9200",
# basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
# ca_certs = 'C:/Users/lenovo/Downloads/elasticsearch-9.1.3-windows-x86_64/elasticsearch-9.1.3/config/certs/http_ca.crt'
# )
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())


{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [2]:
es.index(index="test_index", document={"name": "Kanchan"}, id=1, request_timeout=60)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_17948\4208598054.py:1: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.index(index="test_index", document={"name": "Kanchan"}, id=1, request_timeout=60)


ObjectApiResponse({'_index': 'test_index', '_id': '1', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

## Prepare the data

In [3]:
import pandas as pd
df = pd.read_csv('marathi_dataset.csv')
print(df.shape)
df.head()

(30, 7)


,scheme_id,scheme_name,description,department,category,eligibility,deadline
0,1,शेतकरी अनुदान योजना,बियाण्यांसाठी शेतकऱ्यांना आर्थिक मदत देणारी योजना,कृषी विभाग,अनुदान,जमीन धारक शेतकरी,3/31/2024
1,2,महिला स्वसहाय्य गट योजना,महिला स्वसहाय्य गटांना व्यवसाय सुरू करण्यासाठी...,महिला व बालकल्याण विभाग,कर्ज सबसिडी,महिला स्वसहाय्य गटाचा सदस्य,6/15/2024
2,3,विद्यार्थी शिष्यवृत्ती योजना,शैक्षणिक कामगिरीच्या आधारे विद्यार्थ्यांना शिष...,शिक्षण विभाग,शिष्यवृत्ती,"मागास वर्गीय विद्यार्थी, ७५% पेक्षा जास्त गुण",4/30/2024
3,4,राज्य शासकीय नोकरी अर्ज,राज्य शासकीय सेवेसाठी अर्ज करण्याची ऑनलाइन प्र...,सामान्य प्रशासन विभाग,नोकरी,"१८-३८ वयोगट, संबंधित पात्रता",5/20/2024
4,5,मुफ्त आरोग्य तपासणी शिबिर,ग्रामीण भागात रहिवाशांसाठी विनामूल्य आरोग्य तप...,आरोग्य विभाग,आरोग्य सेवा,सर्व नागरिक,7/10/2024


In [4]:
df.isna().sum()

scheme_id      0
scheme_name    0
description    0
department     0
category       0
eligibility    0
deadline       0
dtype: int64

### Converting description field to vector using sbert model

In [5]:
import numpy as np
print(np.__version__)


2.3.3


In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('l3cube-pune/marathi-sentence-similarity-sbert')

In [7]:
df["description_vector"] = df["description"].apply(lambda x: model.encode(x))

In [8]:
df.head()

,scheme_id,scheme_name,description,department,category,eligibility,deadline,description_vector
0,1,शेतकरी अनुदान योजना,बियाण्यांसाठी शेतकऱ्यांना आर्थिक मदत देणारी योजना,कृषी विभाग,अनुदान,जमीन धारक शेतकरी,3/31/2024,"[-0.019805217, -0.018249715, -0.012199402, 0.0..."
1,2,महिला स्वसहाय्य गट योजना,महिला स्वसहाय्य गटांना व्यवसाय सुरू करण्यासाठी...,महिला व बालकल्याण विभाग,कर्ज सबसिडी,महिला स्वसहाय्य गटाचा सदस्य,6/15/2024,"[-0.026199086, 0.005875388, -0.0020529723, 0.0..."
2,3,विद्यार्थी शिष्यवृत्ती योजना,शैक्षणिक कामगिरीच्या आधारे विद्यार्थ्यांना शिष...,शिक्षण विभाग,शिष्यवृत्ती,"मागास वर्गीय विद्यार्थी, ७५% पेक्षा जास्त गुण",4/30/2024,"[-0.033258807, -0.0118139135, -0.0060276566, 0..."
3,4,राज्य शासकीय नोकरी अर्ज,राज्य शासकीय सेवेसाठी अर्ज करण्याची ऑनलाइन प्र...,सामान्य प्रशासन विभाग,नोकरी,"१८-३८ वयोगट, संबंधित पात्रता",5/20/2024,"[-0.016152378, -0.012807344, 0.019232813, 0.01..."
4,5,मुफ्त आरोग्य तपासणी शिबिर,ग्रामीण भागात रहिवाशांसाठी विनामूल्य आरोग्य तप...,आरोग्य विभाग,आरोग्य सेवा,सर्व नागरिक,7/10/2024,"[-0.013903189, -0.014410155, 0.0020151383, -0...."


In [9]:
es.ping()

True

### create new index in elasticsearch

In [27]:
if es.indices.exists(index="marathi_schemes"):
    es.indices.delete(index="marathi_schemes")
    print("Old index deleted ✅")

from elasticSearch.indexMappings import indexMappings
es.indices.create(index="marathi_schemes", mappings=indexMappings , request_timeout=120)
print("New index created successfully 🎉")


Old index deleted ✅


C:\Users\lenovo\AppData\Local\Temp\ipykernel_17948\2726722812.py:6: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index="marathi_schemes", mappings=indexMappings , request_timeout=120)


New index created successfully 🎉


### insert data into index

In [28]:
record_list = df.to_dict("records")

In [29]:
record_list[8]

{'scheme_id': 9,
 'scheme_name': 'मोफत इंटरनेट सेवा',
 'description': 'शाळा-महाविद्यालयांमध्ये मोफत इंटरनेट सेवा पुरवठा',
 'department': 'तंत्रज्ञान विभाग',
 'category': 'सुविधा',
 'eligibility': 'शासकीय शैक्षणिक संस्था',
 'deadline': '9/1/2024',
 'description_vector': array([-3.28198932e-02, -1.22763366e-02, -9.14190989e-03, -1.26465010e-02,
         2.37572677e-02, -3.38627957e-02,  1.34045957e-03, -3.57055361e-03,
        -1.65964831e-02,  1.39760133e-02,  2.09304057e-02, -2.56308191e-03,
         1.75180901e-02, -9.02399793e-03, -7.20589515e-03, -1.53395403e-02,
        -3.63764848e-04, -6.04298245e-03,  1.57842580e-02, -8.94714706e-03,
         1.24763446e-02,  1.20191388e-02, -1.53071489e-02,  1.44403027e-02,
         1.84102952e-02, -1.46792829e-02, -6.14943868e-03,  1.93558692e-03,
        -9.39143170e-03,  1.02639608e-02,  1.71703126e-04,  8.83889385e-03,
         9.84950829e-03,  2.73562856e-02,  1.17094745e-03,  4.79097813e-02,
        -1.33460062e-02, -2.13035736e-02,  2.14

In [35]:
es.cluster.health(wait_for_status="yellow")
print("Cluster ready ✅")


ApiError: ApiError(408, "{'cluster_name': 'elasticsearch', 'status': 'red', 'timed_out': True, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 4, 'active_shards': 4, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 3, 'unassigned_primary_shards': 1, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 57.14285714285714}")

### convert all records into index in elastic search

In [34]:
es.count(index="marathi_schemes")

ApiError: ApiError(503, 'search_phase_execution_exception', None)

## search in data

In [33]:
input_text = "विद्यार्थी"
vector = model.encode(input_text)
print(len(vector))

query = {
    "size": 2,
    "knn": {
        "field": "description_vector",
        "query_vector": vector.tolist(),
        "k": 2,
        "num_candidates": 10
    }
}

result = es.search(
    index="marathi_schemes",
    body={
        **query,
        "_source": ["scheme_name", "description"]
    }
)

for hit in result['hits']['hits']:
    print(f"Score: {hit['_score']:.4f}")
    print(f"Scheme: {hit['_source']['scheme_name']}")
    print(f"Description: {hit['_source']['description']}\n")


768


ApiError: ApiError(503, 'search_phase_execution_exception', None)

In [32]:
mapping = es.indices.get_mapping(index="marathi_schemes")
print(mapping["marathi_schemes"]["mappings"]["properties"].keys())


dict_keys(['category', 'deadline', 'department', 'description', 'description_vector', 'eligibility', 'scheme_id', 'scheme_name'])


In [24]:
res = es.search(
    index="marathi_schemes",
    query={"match": {"description": "विद्यार्थी"}},
    size=3,
    _source=["scheme_name", "description"]
)

for hit in res["hits"]["hits"]:
    print(hit["_source"]["scheme_name"])


In [31]:
print(es.count(index="marathi_schemes")["count"])

ApiError: ApiError(503, 'search_phase_execution_exception', None)